# LangChain RAG Agent Workshop
## Build a Document-Aware Agent Step by Step
**What we'll build:** An AI agent that can search your documents and answer questions.

## What is LangChain? Understanding the Ecosystem

**LangChain** is a framework for building applications powered by large language models (LLMs). Rather than one monolithic library, LangChain is split into focused packages — each with a clear responsibility.

### The Core Packages

| Package | What It Does | Example |
|---------|-------------|---------|
| **`langchain-core`** | Base abstractions shared by everything — `Runnable`, `Document`, `ChatPromptTemplate`, `tool` decorator. Installed automatically. | `from langchain_core.tools import tool` |
| **`langchain`** | High-level chains, agents, and orchestration logic that glue components together. | `from langchain.agents import create_agent` |
| **`langchain-community`** | Community-maintained integrations (vector stores, loaders, retrievers) that don't have their own package yet. | `from langchain_community.vectorstores import FAISS` |

![Generated Image March 26, 2026 - 11_01PM.jpg](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_2_Generated_Image_March_26_2026_-_11_01PM.jpg?raw=true)

---
# Section 1: Setup & Core Components
---

## 1.1 Install Packages

We need three layers:
- **LLM provider** (`langchain-google-genai`) - talks to Google's Gemini
- **Local embeddings** (`langchain-huggingface`) - runs on your machine
- **Document parsing** (`pypdf`) - reads PDF files

In [1]:
# ============================================
# Run this cell FIRST - installs everything
# Works on: Google Colab, Kaggle, local Jupyter
# ============================================

import subprocess, sys

PACKAGES = [
    "langchain",
    "langchain-google-genai",
    "langchain-huggingface",
    "langchain-community",
    "faiss-cpu",
    "pypdf",
    "sentence-transformers",
    "python-dotenv",
]

# Step 1: Ensure pip exists
try:
    subprocess.run(
        [sys.executable, "-m", "ensurepip", "--upgrade"],
        capture_output=True,
    )
except Exception:
    pass

# Step 2: Install packages
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--upgrade", "pip"],
    capture_output=True,
)
result = subprocess.run(
    [sys.executable, "-m", "pip", "install"] + PACKAGES,
    capture_output=True,
    text=True,
)
if result.returncode != 0:
    print("Installation error:")
    print(result.stderr)
else:
    print("Packages installed.")

# Step 3: Verify every import actually works
IMPORTS_TO_VERIFY = [
    ("langchain", "langchain"),
    ("langchain_google_genai", "langchain-google-genai"),
    ("langchain_huggingface", "langchain-huggingface"),
    ("langchain_community", "langchain-community"),
    ("faiss", "faiss-cpu"),
    ("pypdf", "pypdf"),
    ("sentence_transformers", "sentence-transformers"),
    ("dotenv", "python-dotenv"),
]

all_good = True
for module_name, package_name in IMPORTS_TO_VERIFY:
    try:
        __import__(module_name)
        print(f"  {package_name:30s} OK")
    except ImportError:
        print(f"  {package_name:30s} FAILED")
        all_good = False

print()
if all_good:
    print("All set! You are ready to go.")
else:
    print("Some imports failed.")
    print("In Colab: Runtime > Restart runtime, then re-run this cell.")
    print("Locally: make sure your Jupyter kernel matches your venv.")


Packages installed.
  langchain                      OK
  langchain-google-genai         OK
  langchain-huggingface          OK
  langchain-community            OK
  faiss-cpu                      OK
  pypdf                          OK
  sentence-transformers          OK
  python-dotenv                  OK

All set! You are ready to go.


## 1.2 Load Your API Key

**Get your key:**
1. Go to https://aistudio.google.com/app/apikeys
2. Click **Create API Key** → **Create API Key in new project**
3. Copy and save it somewhere safe

### Running on Google Colab?

Colab does not support `.env` files. Use **Colab Secrets** instead:

1. Click the **key icon** in the left sidebar
2. Click **"Add new secret"**
3. Set the name to `GOOGLE_API_KEY` and paste your key as the value
4. Toggle **"Notebook access"** ON

The cell below will automatically detect which environment you are in.

In [34]:
import os

# --- Try each method in order ---

# Method 1: Colab Secrets (Google Colab)
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    print("API key loaded from Colab Secrets.")
except (ImportError, ModuleNotFoundError):
    # Method 2: .env file (local Jupyter)
    from dotenv import load_dotenv
    load_dotenv()

# Verify
if os.getenv("GOOGLE_API_KEY"):
    print("API key loaded!")
else:
    print("No API key found.")
    print("  - Colab: Add GOOGLE_API_KEY to Secrets (key icon in sidebar)")
    print("  - Local: Create a .env file with GOOGLE_API_KEY=your-key-here")


API key loaded!


![Generated Image March 26, 2026 - 11_06PM.jpg](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_8_Generated_Image_March_26_2026_-_11_06PM.jpg?raw=true)

## 1.3 Test the Embedding Model

**Key Concept:** Embeddings convert text into vectors (lists of numbers). Similar texts have similar vectors.

We use `all-MiniLM-L6-v2` because:
- Fast (small model)
- Good quality for its size
- Runs 100% locally (no API calls)

In [35]:
from langchain_huggingface import HuggingFaceEmbeddings

# Initialize (downloads model on first run)
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model ready!


### Try It: Test Embedding Similarity

In [36]:
# Embed two sentences
text1 = "The cat sat on the mat"
text2 = "A kitten rested on the rug"
text3 = "Python is a programming language"
text4 = "I hate shawrma"
text5 = "I don't hate shawrma"
vec1 = embeddings.embed_query(text1)
vec2 = embeddings.embed_query(text2)
vec3 = embeddings.embed_query(text3)
vec4 = embeddings.embed_query(text4)
vec5 = embeddings.embed_query(text5)
print(f"Vector length: {len(vec1)} dimensions")
print(f"First 5 values: {vec1[:5]}")

Vector length: 384 dimensions
First 5 values: [0.1304018348455429, -0.011870122514665127, -0.028116989880800247, 0.05123864859342575, -0.05597448721528053]


![Generated Image March 26, 2026 - 11_06PM.jpg](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_13_Generated_Image_March_26_2026_-_11_06PM.jpg?raw=true)

In [37]:
# Calculate similarity (dot product)
def similarity(v1, v2):
    return sum(a * b for a, b in zip(v1, v2))

print(f"'cat on mat' vs 'kitten on rug': {similarity(vec1, vec2):.3f}")
print(f"'cat on mat' vs 'Python language': {similarity(vec1, vec3):.3f}")
print("i hate shawrma vs i don't hate shawrma " , similarity(vec4,vec5))
print(f"\nHigher = more similar!")

'cat on mat' vs 'kitten on rug': 0.613
'cat on mat' vs 'Python language': 0.031
i hate shawrma vs i don't hate shawrma  0.9738682310660554

Higher = more similar!


# Limitation:
As we can see vector 4 and vector 5 are the opposite of each other , when it comes to meaning but this isn't reflected in the similarity score why?
well when the words are projected to the dimensions they are close to each other since they match in most of structure , which showcases on the most challenging flaws in embeddings.
so how can we mitigate it ?

---
# Section 2: Data Ingestion & Retrieval (RAG)
---

![Generated Image March 26, 2026 - 11_09PM.jpg](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_17_Generated_Image_March_26_2026_-_11_09PM.jpg?raw=true)

## 2.1 Sample Document

We'll use a sample text document for this workshop. In production, you'd load from PDF, Word, etc.

**Note:** If your PDF gives empty content (common with "Print to PDF"), use a text-based approach like below.

In [38]:
# Sample document content (workshop prerequisites)
SAMPLE_DOCUMENT = """
# Prerequisites for LangChain RAG Agent Workshop

Before starting the workshop, you need to set up three things: Python, pip (package manager), and Google API credentials.

## P.1 Install Python

Python is the programming language that runs everything in this workshop. It's like the engine of a car - without it, nothing works.

Why Python?
- LangChain is built in Python - we need Python to use it
- AI/ML ecosystem - most AI tools (PyTorch, TensorFlow, LangChain) are written in Python
- Easy to learn - clean syntax makes it beginner-friendly

How to Install Python on Windows:
1. Go to python.org
2. Click the yellow "Download Python" button
3. Run the installer
4. IMPORTANT: Check the box that says "Add Python to PATH"
5. Click Install Now
6. Verify it worked by running: python --version

You should see something like Python 3.11.x or similar.

## P.2 Package Manager (pip)

Python comes with pip (Python's built-in package manager) already installed. Pip is like a librarian for code - it handles downloading and installing libraries.

Verify pip is installed:
pip --version

You should see something like pip 24.x or higher.

## P.3 Create a Virtual Environment

A virtual environment is an isolated folder on your computer where you install project-specific packages. Think of it like a separate workspace that keeps your workshop dependencies separate from other Python projects.

Why use a virtual environment?
- Isolation - If you have multiple Python projects, they won't interfere with each other
- Cleanliness - You can delete the entire folder later without affecting your system Python
- Easy debugging - If something breaks, you know it's in this project, not your whole system
- Best practice - Professional developers always use virtual environments

How to Create and Activate:
1. Open your terminal/PowerShell and navigate to your workshop folder
2. Create the virtual environment: python -m venv venv
3. Activate it on Windows: .\\venv\\Scripts\\Activate.ps1
4. Activate it on macOS/Linux: source venv/bin/activate

You'll see (venv) appear at the start of your terminal line - that means you're inside the virtual environment.

## P.4 Install Required Packages

You're installing libraries (pre-built code) that this workshop needs:
- langchain-core - Core LangChain functionality
- langchain-google-genai - Connector for Google's Gemini API
- langchain-huggingface - Local embedding models
- langchain-community - Additional LangChain tools
- faiss-cpu - Fast similarity search for vectors
- pypdf - Read PDF documents
- sentence-transformers - Power the embeddings
- python-dotenv - Load your API key securely from .env

Install command:
pip install langchain-core langchain-google-genai langchain-huggingface langchain-community faiss-cpu pypdf sentence-transformers python-dotenv

This will take a few minutes - pip is downloading and installing everything.

## P.5 Create a Google Account and Get API Key

An API key is like a password for using Google's AI services. It tells Google's servers "this is me, let me use Gemini."

Why Google Gemini?
- Free tier available - you get free API calls for learning (generous limits)
- Fast and capable - gemini-2.0-flash is quick enough for RAG tasks
- Simple setup - easier than OpenAI for workshops
- No credit card required (unless you exceed free tier)

Step 1: Create a Google Account (if you don't have one)
Step 2: Get Your API Key
1. Go to Google AI Studio (aistudio.google.com)
2. Click "Create API Key" button
3. Select "Create API Key in new project"
4. Google generates a key for you instantly
5. Copy the key (you won't see it again - save it!)

Security Reminder:
- Never share your API key publicly - anyone with it can use your free tier quota
- Never commit it to GitHub - use .env files instead

## Troubleshooting

"Python command not found"
- Windows: You skipped checking "Add Python to PATH" during installation. Reinstall and check that box.
- macOS/Linux: Use python3 instead of python

"pip is not recognized" (Windows)
- Make sure you've activated the virtual environment first
- Look for (venv) at the start of your terminal line

"pip install is taking forever"
- This is normal! Installing torch and sentence-transformers can take 5-15 minutes.
- Let it run. Don't interrupt it.

"ModuleNotFoundError" when running workshop code
- Make sure the virtual environment is activated: (venv) should show in your terminal
- Reinstall the packages if needed

Once everything is installed and verified, you're ready for the workshop!
"""

print(f"Sample document loaded: {len(SAMPLE_DOCUMENT)} characters")

Sample document loaded: 4527 characters


In [39]:
# Convert to LangChain Document format
from langchain_core.documents import Document

documents = [Document(
    page_content=SAMPLE_DOCUMENT,
    metadata={"source": "workshop_prerequisites", "type": "guide"}
)]

print(f"Created {len(documents)} document(s)")
print(f"Metadata: {documents[0].metadata}")

Created 1 document(s)
Metadata: {'source': 'workshop_prerequisites', 'type': 'guide'}


### Alternative: Load from PDF

If you have a PDF that works, use this instead:

In [40]:
# Uncomment to load from PDF instead:
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader("your_document.pdf")
# documents = loader.load()
# print(f"Loaded {len(documents)} pages from PDF")

### Discussion Question
> What information might get lost when converting PDFs to text?
>
> Think about: tables, images, formatting, headers/footers...

## 2.2 Split Text into Chunks

**Key Concept:** We split documents into smaller pieces because:
1. LLMs have context limits
2. Smaller chunks = more precise retrieval
3. Embedding quality is better on focused text

**RecursiveCharacterTextSplitter** tries to split at natural boundaries:
- First tries: `\n\n` (paragraphs)
- Then: `\n` (lines)
- Then: ` ` (words)
- Finally: characters

![Generated Image March 26, 2026 - 11_14PM (1).jpg](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_25_Generated_Image_March_26_2026_-_11_14PM_1.jpg?raw=true)

In [41]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     # Target size per chunk
    chunk_overlap=50    # Overlap between chunks
)

chunks = splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

Split into 12 chunks


In [42]:
# Look at a few chunks
for i, chunk in enumerate(chunks[:3]):
    print(f"--- Chunk {i} ({len(chunk.page_content)} chars) ---")
    print(chunk.page_content)
    print()

--- Chunk 0 (328 chars) ---
# Prerequisites for LangChain RAG Agent Workshop

Before starting the workshop, you need to set up three things: Python, pip (package manager), and Google API credentials.

## P.1 Install Python

Python is the programming language that runs everything in this workshop. It's like the engine of a car - without it, nothing works.

--- Chunk 1 (466 chars) ---
Why Python?
- LangChain is built in Python - we need Python to use it
- AI/ML ecosystem - most AI tools (PyTorch, TensorFlow, LangChain) are written in Python
- Easy to learn - clean syntax makes it beginner-friendly

How to Install Python on Windows:
1. Go to python.org
2. Click the yellow "Download Python" button
3. Run the installer
4. IMPORTANT: Check the box that says "Add Python to PATH"
5. Click Install Now
6. Verify it worked by running: python --version

--- Chunk 2 (375 chars) ---
You should see something like Python 3.11.x or similar.

## P.2 Package Manager (pip)

Python comes with pip (Python's

### Experiment: Chunk Size Matters!

In [43]:
# Try different chunk sizes
for size in [200, 500, 1000, 2000]:
    test_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    test_chunks = test_splitter.split_documents(documents)
    print(f"chunk_size={size}: {len(test_chunks)} chunks")

chunk_size=200: 33 chunks
chunk_size=500: 12 chunks
chunk_size=1000: 5 chunks
chunk_size=2000: 3 chunks


## 2.3 Create a Vector Store

**Key Concept:** A vector store is a searchable index of embeddings.

| Store | Persistence | Best For |
|-------|-------------|----------|
| FAISS | In-memory | Prototyping, small datasets |
| Chroma | Local file | Persistent local storage |
| Pinecone | Cloud | Production, billions of docs |

![Generated Image March 26, 2026 - 11_12PM (1).jpg](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_31_Generated_Image_March_26_2026_-_11_12PM_1.jpg?raw=true)

In [44]:
from langchain_community.vectorstores import FAISS

# Create the vector store (embeds all chunks)
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"Vector store created with {len(chunks)} chunks!")

Vector store created with 12 chunks!


In [45]:
# Test a similarity search directly
query = "How do I install Python?"
results = vectorstore.similarity_search(query, k=3)

print(f"Query: {query}\n")
for i, doc in enumerate(results):
    print(f"Result {i+1}:")
    print(doc.page_content[:200] + "...")
    print()

Query: How do I install Python?

Result 1:
You should see something like Python 3.11.x or similar.

## P.2 Package Manager (pip)

Python comes with pip (Python's built-in package manager) already installed. Pip is like a librarian for code - i...

Result 2:
Why Python?
- LangChain is built in Python - we need Python to use it
- AI/ML ecosystem - most AI tools (PyTorch, TensorFlow, LangChain) are written in Python
- Easy to learn - clean syntax makes it b...

Result 3:
Install command:
pip install langchain-core langchain-google-genai langchain-huggingface langchain-community faiss-cpu pypdf sentence-transformers python-dotenv

This will take a few minutes - pip is ...



![image.png](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_34_image.png?raw=true)

## 2.4 Create a Retriever

**Key Concept:** A retriever wraps the vector store. This abstraction lets you swap storage backends without changing your code.

The retriever will become a **tool** for our agent!

In [46]:
retriever = vectorstore.as_retriever(
    search_type="similarity",  # or "mmr" for diversity
    search_kwargs={"k": 3}     # return top 3 results
)

print("Retriever created!")

Retriever created!


In [47]:
# Test the retriever
docs = retriever.invoke("What is a virtual environment?")

print(f"Retrieved {len(docs)} documents")
print(f"\nFirst result: {docs[0].page_content}")

Retrieved 3 documents

First result: ## P.3 Create a Virtual Environment

A virtual environment is an isolated folder on your computer where you install project-specific packages. Think of it like a separate workspace that keeps your workshop dependencies separate from other Python projects.


---
# Section 3: Building the Agent
---

## 3.1 Define Output Schema (Pydantic)

**Key Concept:** Pydantic ensures the LLM's response has a predictable structure.

This is useful when:
- Downstream code expects specific fields
- You need validation
- You want type safety

In [48]:
from pydantic import BaseModel, Field

class RAGAnswer(BaseModel):
    """Structured answer from the RAG agent."""
    
    answer: str = Field(
        description="The answer to the user's question"
    )
    confidence: str = Field(
        description="How confident: 'high', 'medium', or 'low'"
    )
    source_found: bool = Field(
        description="Whether relevant info was found in documents"
    )

print("Schema defined!")
print(RAGAnswer.model_json_schema())

Schema defined!
{'description': 'Structured answer from the RAG agent.', 'properties': {'answer': {'description': "The answer to the user's question", 'title': 'Answer', 'type': 'string'}, 'confidence': {'description': "How confident: 'high', 'medium', or 'low'", 'title': 'Confidence', 'type': 'string'}, 'source_found': {'description': 'Whether relevant info was found in documents', 'title': 'Source Found', 'type': 'boolean'}}, 'required': ['answer', 'confidence', 'source_found'], 'title': 'RAGAnswer', 'type': 'object'}


### Discussion Question
> Why is structured output better than plain text for downstream systems?
>
> Think about: parsing, validation, error handling...

## 3.2 Wrap the Retriever as a Tool

**Key Concept:** Tools are how agents interact with external systems.

The `@tool` decorator tells the agent:
- What the tool does (from docstring)
- What inputs it expects
- What it returns

![Generated Image March 26, 2026 - 11_18PM.jpg](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_43_Generated_Image_March_26_2026_-_11_18PM.jpg?raw=true)

In [49]:
from langchain_core.tools import tool

@tool
def search_documents(query: str) -> str:
    """Search the knowledge base for relevant information.
    
    Use this tool to find answers in the loaded documents.
    Returns relevant text chunks that may contain the answer.
    """
    results = retriever.invoke(query)
    if not results:
        return "No relevant documents found."
    return "\n\n---\n\n".join([doc.page_content for doc in results])

# Put tools in a list
tools = [search_documents]

print("Tool created!")
print(f"Name: {search_documents.name}")
print(f"Description: {search_documents.description}")

Tool created!
Name: search_documents
Description: Search the knowledge base for relevant information.

Use this tool to find answers in the loaded documents.
Returns relevant text chunks that may contain the answer.


In [50]:
# Test the tool directly
result = search_documents.invoke("How do I get an API key?")
print(result)

Step 1: Create a Google Account (if you don't have one)
Step 2: Get Your API Key
1. Go to Google AI Studio (aistudio.google.com)
2. Click "Create API Key" button
3. Select "Create API Key in new project"
4. Google generates a key for you instantly
5. Copy the key (you won't see it again - save it!)

Security Reminder:
- Never share your API key publicly - anyone with it can use your free tier quota
- Never commit it to GitHub - use .env files instead

## Troubleshooting

---

Install command:
pip install langchain-core langchain-google-genai langchain-huggingface langchain-community faiss-cpu pypdf sentence-transformers python-dotenv

This will take a few minutes - pip is downloading and installing everything.

## P.5 Create a Google Account and Get API Key

An API key is like a password for using Google's AI services. It tells Google's servers "this is me, let me use Gemini."

---

## P.4 Install Required Packages

You're installing libraries (pre-built code) that this workshop needs:

## 3.3 Create a Prompt Template

**Key Concept:** `ChatPromptTemplate.from_messages` lets you define:
- System instructions
- Example conversations (few-shot)
- User input placeholder

In [51]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful document assistant.

RULES:
1. Always use the search_documents tool before answering
2. If you can't find the answer, say so honestly
3. Cite which part of the document you used
4. Keep answers concise but complete"""),
    
    # Few-shot example (optional but helpful)
    ("user", "What is this document about?"),
    ("assistant", "Let me search for that information."),
    
    # Actual user input goes here
    ("user", "{input}")
])

print("Prompt template created!")

Prompt template created!


In [52]:
# Preview what the prompt looks like
formatted = prompt.invoke({"input": "How do I install Python?"})

for msg in formatted.messages:
    print(f"[{msg.type.upper()}]")
    print(msg.content)
    print()

[SYSTEM]
You are a helpful document assistant.

RULES:
1. Always use the search_documents tool before answering
2. If you can't find the answer, say so honestly
3. Cite which part of the document you used
4. Keep answers concise but complete

[HUMAN]
What is this document about?

[AI]
Let me search for that information.

[HUMAN]
How do I install Python?



## 3.4 Create the Agent

**Key Concept:** The agent is the orchestrator. It:
1. Receives user input
2. Decides which tools to call
3. Calls tools and reads results
4. Generates final answer

The model string format: `provider:model_name`

In [53]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=tools,
    response_format=RAGAnswer,
    system_prompt="""You are a helpful document assistant.
    
    
Always use the search_documents tool to find information before answering.
If the answer isn't in the documents, say so clearly.
Keep answers concise but complete."""
)

print("Agent created!")

Agent created!


## 3.5 Run and Test the Agent

In [54]:
# Helper function to chat with the agent
def ask(question: str) -> str:
    """Send a question to the agent and get the answer."""
    response = agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    return response["structured_response"]

In [55]:
# Test question 1
print("Q: How do I install Python on Windows?")
result = ask("How do I install Python on Windows?")
print(result.answer)
print(result.confidence)
print(result.source_found)


Q: How do I install Python on Windows?
To install Python on Windows, follow these steps:

1. Go to [python.org](https://www.python.org).
2. Click the yellow "Download Python" button.
3. Run the installer once it has finished downloading.
4. **Important:** Check the box that says "Add Python to PATH" before proceeding.
5. Click "Install Now."
6. Verify the installation by opening a terminal and running the command: `python --version` (you should see something like Python 3.11.x).

If you get a "Python command not found" error, you likely skipped checking the "Add Python to PATH" box and may need to reinstall with that option selected.
high
True


In [56]:
# Test question 2
print("Q: What is a virtual environment and why should I use one?")
print("A:", ask("What is a virtual environment and why should I use one?"))

Q: What is a virtual environment and why should I use one?
A: answer="A virtual environment is an isolated folder on your computer where you install project-specific packages and dependencies, acting as a separate workspace from your global system. \n\nYou should use one because:\n- **Isolation:** It prevents different projects from interfering with each other's specific requirements.\n- **Cleanliness:** You can delete the project and its dependencies easily without affecting your system-wide Python installation.\n- **Easier Debugging:** If something breaks, the issue is contained within that specific project environment.\n- **Best Practice:** It is a professional standard that keeps your development environment organized and reproducible." confidence='high' source_found=True


In [57]:
# Test question 3 - something NOT in the document
print("Q: How do I deploy to AWS?")
print("A:", ask("How do I deploy to AWS?"))

Q: How do I deploy to AWS?
A: answer="I'm sorry, but I couldn't find any information on how to deploy to AWS in the provided documents. The documents focus on setting up a Google AI Studio account, managing API keys, and configuring Python virtual environments." confidence='high' source_found=False


---
# Section 4: Testing & Exploration
---

## 4.1 Testing Checklist

Run through these tests:

In [58]:
# Test 1: Does the tool get called?
print("=" * 50)
print("Test 1: Basic question")
print("=" * 50)
print(ask("What packages do I need to install?"))

Test 1: Basic question
answer='To set up your project, you need to install the following packages using pip:\n\n- `langchain-core`\n- `langchain-google-genai`\n- `langchain-huggingface`\n- `langchain-community`\n- `faiss-cpu`\n- `pypdf`\n- `sentence-transformers`\n- `python-dotenv` \n\nYou can install them all at once using the command:\n`pip install langchain-core langchain-google-genai langchain-huggingface langchain-community faiss-cpu pypdf sentence-transformers python-dotenv`' confidence='high' source_found=True


In [59]:
# Test 2: Rephrased question
print("=" * 50)
print("Test 2: Rephrased question")
print("=" * 50)
print(ask("What libraries are required for this workshop?"))

Test 2: Rephrased question
answer="The following libraries are required for the workshop:\n- **langchain-core**: Core LangChain functionality\n- **langchain-google-genai**: Connector for Google's Gemini API\n- **langchain-huggingface**: Local embedding models\n- **langchain-community**: Additional LangChain tools\n- **faiss-cpu**: Fast similarity search for vectors\n- **pypdf**: To read PDF documents\n- **sentence-transformers**: To power the embeddings\n- **python-dotenv**: To load API keys securely from a .env file" confidence='high' source_found=True


In [60]:
# Test 3: Question outside the document
print("=" * 50)
print("Test 3: Out-of-scope question")
print("=" * 50)
print(ask("What's the weather like today?"))

Test 3: Out-of-scope question
answer="I'm sorry, I don't have information about today's weather in the provided documents. My context is focused on the prerequisites and troubleshooting for a LangChain RAG Agent workshop." confidence='high' source_found=False


In [61]:
# Test 4: Troubleshooting question
print("=" * 50)
print("Test 4: Troubleshooting")
print("=" * 50)
print(ask("pip install is taking forever, is this normal?"))

Test 4: Troubleshooting
answer='Yes, this is normal. According to the documentation, installing packages like `torch` and `sentence-transformers` can take between 5 to 15 minutes. It is recommended to let the process run and avoid interrupting it, as pip is downloading and installing multiple libraries simultaneously.' confidence='high' source_found=True


## 4.2 Bonus: Add a Second Tool

In [65]:
@tool
def list_document_sections() -> str:
    """List the main sections/topics in the document.
    
    Use this to understand what topics the document covers
    before searching for specific information.
    """
    sections = []
    for line in SAMPLE_DOCUMENT.split("\n"):
        if line.startswith("## "):
            sections.append(line.replace("## ", ""))
    return "Document sections:\n" + "\n".join(f"- {s}" for s in sections)

# Test it
print(list_document_sections.invoke(""))

Document sections:
- P.1 Install Python
- P.2 Package Manager (pip)
- P.3 Create a Virtual Environment
- P.4 Install Required Packages
- P.5 Create a Google Account and Get API Key
- Troubleshooting


In [66]:
# Create agent with both tools

multi_tool_agent = create_agent(
    model="google_genai:gemini-3-flash-preview",
    tools=[search_documents, list_document_sections],
    response_format = RAGAnswer,
    system_prompt="""You are a helpful document assistant with two tools:
- search_documents: Search for specific information
- list_document_sections: Get an overview of document structure

Choose the right tool based on the question."""
)

print("Multi-tool agent created!")

Multi-tool agent created!


In [64]:
# Test the multi-tool agent
def ask_multi(question: str) -> str:
    response = multi_tool_agent.invoke({
        "messages": [{"role": "user", "content": question}]
    })
    return response['structured_response']

print("Q: What topics does this document cover?")
result = ask_multi("What topics does this document cover?")
print(result.answer)
print(result.confidence)

Q: What topics does this document cover?
The document covers the following topics:
- Installing Python
- Using the Package Manager (pip)
- Creating a Virtual Environment
- Installing Required Packages
- Creating a Google Account and obtaining an API Key
- Troubleshooting
high


In [67]:
# Now a specific question
print("Q: How do I fix 'pip is not recognized' error?")
print("A:", ask_multi("How do I fix 'pip is not recognized' error?"))

Q: How do I fix 'pip is not recognized' error?
A: answer="To fix the 'pip is not recognized' error on Windows, ensure that you have activated your virtual environment first. You can confirm it is active by looking for '(venv)' at the start of your terminal line. \n\nAdditionally, if you encounter a 'Python command not found' error, it may be because 'Add Python to PATH' was skipped during installation; in that case, reinstalling and checking that box is recommended." confidence='high' source_found=True


![image.png](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_67_image.png?raw=true)

---
## Quick Reference

| Component | Purpose | Key Function |
|-----------|---------|-------------|
| Loader | Read documents | `PyPDFLoader().load()` |
| Splitter | Chunk text | `splitter.split_documents()` |
| Embeddings | Text → vectors | `embeddings.embed_query()` |
| Vector Store | Similarity search | `FAISS.from_documents()` |
| Retriever | Fetch chunks | `retriever.invoke()` |
| Tool | Agent capability | `@tool` decorator |
| Agent | Orchestrator | `create_agent()` |

---

## Reflection Questions

1. Where does the LLM's intelligence come into play vs. simple retrieval?
2. What's the cost/benefit of local embeddings vs. cloud embeddings?
3. How would you debug if the agent isn't finding the right documents?
4. Why is the retriever abstraction important for production?
5. How would switching vector stores affect your agent?

---

![Generated Image March 26, 2026 - 11_28PM.jpg](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_69_Generated_Image_March_26_2026_-_11_28PM.jpg?raw=true)

![image.png](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_70_image.png?raw=true)

![image.png](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_71_image.png?raw=true)

![Generated Image March 26, 2026 - 11_33PM.jpg](https://github.com/h19overflow/Langchain_Workshop/blob/master/materials/images/cell_72_Generated_Image_March_26_2026_-_11_33PM.jpg?raw=true)

---
# 📚 Continue Your Journey — Resources & Documentation

Your learning doesn't stop here. Below is a progressive list of resources — start from the top and work your way down as you grow.

---

## Level 1: Foundations (Start Here)

| Resource | What You'll Learn | Link |
|----------|------------------|------|
| **LangChain Docs** | Core concepts — chains, prompts, tools, agents | [python.langchain.com](https://python.langchain.com/docs/introduction/) |
| **LangChain Academy** | Free structured courses from the LangChain team | [academy.langchain.com](https://academy.langchain.com/) |
| **LangChain Hub** | Browse and reuse community prompts | [smith.langchain.com/hub](https://smith.langchain.com/hub) |

---

## Level 2: Better Retrieval & Document Processing

| Resource | What You'll Learn | Link |
|----------|------------------|------|
| **LangChain RAG Tutorial** | End-to-end RAG with different retrieval strategies | [RAG Tutorial](https://python.langchain.com/docs/tutorials/rag/) |
| **Text Splitters Guide** | Recursive, semantic, markdown, and code chunkers | [Text Splitters](https://python.langchain.com/docs/concepts/text_splitters/) |
| **MTEB Leaderboard** | Compare embedding models by language and task | [huggingface.co/spaces/mteb/leaderboard](https://huggingface.co/spaces/mteb/leaderboard) |
| **FAISS Documentation** | Vector indexing and similarity search at scale | [faiss.ai](https://faiss.ai/) |
| **Sentence Transformers** | Local embedding models (multilingual, cross-encoders for re-ranking) | [sbert.net](https://www.sbert.net/) |

---

## Level 3: Evaluation & Quality

| Resource | What You'll Learn | Link |
|----------|------------------|------|
| **RAGAS** | Open-source RAG evaluation — faithfulness, relevancy, correctness | [ragas.io](https://docs.ragas.io/) |
| **DeepEval** | Unit testing framework for LLM outputs | [deepeval.com](https://docs.confident-ai.com/) |
| **LangSmith** | Tracing, evaluation, and dataset management for LangChain apps | [smith.langchain.com](https://smith.langchain.com/) |

---

## Level 4: Agentic Architectures

| Resource | What You'll Learn | Link |
|----------|------------------|------|
| **LangGraph Docs** | Build stateful multi-agent workflows with cycles and branching | [langchain-ai.github.io/langgraph](https://langchain-ai.github.io/langgraph/) |
| **LangGraph Academy Course** | Free course — ReAct agents, multi-agent, human-in-the-loop | [LangGraph Course](https://academy.langchain.com/courses/intro-to-langgraph) |
| **LangGraph Templates** | Pre-built agent architectures you can fork | [Templates](https://langchain-ai.github.io/langgraph/tutorials/workflows/) |

---

## Level 5: Observability & Production

| Resource | What You'll Learn | Link |
|----------|------------------|------|
| **LangFuse** | Open-source LLM observability — traces, scores, prompt management | [langfuse.com/docs](https://langfuse.com/docs) |
| **Arize Phoenix** | Open-source tracing and retrieval evaluation | [arize.com/phoenix](https://docs.arize.com/phoenix) |
| **LangServe** | Deploy LangChain chains as REST APIs | [LangServe Docs](https://python.langchain.com/docs/langserve/) |

---

## Level 6: Advanced Topics & Research

| Resource | What You'll Learn | Link |
|----------|------------------|------|
| **Cohere Rerank** | Production re-ranking API for retrieval precision | [cohere.com/rerank](https://docs.cohere.com/docs/rerank-2) |
| **CAMeL Tools** | Arabic NLP — morphology, dialect ID, diacritization | [camel-tools.readthedocs.io](https://camel-tools.readthedocs.io/) |
| **Unstructured.io** | Universal document parsing (PDF, DOCX, images, OCR) | [unstructured.io](https://docs.unstructured.io/) |
| **LangChain Experimental** | Bleeding-edge features — semantic chunking, autonomous agents | [langchain-experimental](https://python.langchain.com/docs/integrations/providers/langchain_experimental/) |

---

> **Tip:** Don't try to learn everything at once. Pick one level, build something small, evaluate it, then move to the next. The best way to learn is to **build → measure → improve**.